# CosyVoice3 Fine-Tuning (Debi & Marlene) - Colab A100

Base Model: `FunAudioLLM/Fun-CosyVoice3-0.5B-2512`

- LLM SFT only (Flow/HiFiGAN pretrained 유지)
- Multi-speaker: debi + marlene
- 4 style tags: `[excited]`, `[sad]`, `[happy]`, `[calm]`
- Colab 시스템 Python + PyTorch 그대로 사용 (conda 없음)

공식 파이프라인 (`examples/libritts/cosyvoice3/run.sh`) 기반

---
## 1. GPU 확인 + Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

import torch, os
print(f'\ntorch {torch.__version__} CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

WORK_DIR = '/content/CosyVoice'
DATA_DIR = '/content/cosyvoice3_data'
BACKUP_DIR = '/content/drive/MyDrive/cosyvoice3_backup'
os.makedirs(BACKUP_DIR, exist_ok=True)

---
## 2. 데이터 업로드 (Drive zip)

In [ ]:
import os, shutil

DATA_DIR = '/content/cosyvoice3_data'
DRIVE_ZIP = '/content/drive/MyDrive/cosyvoice3_data.zip'

if os.path.exists(DRIVE_ZIP):
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    !unzip -q "{DRIVE_ZIP}" -d /content/
    print(f'Unzipped: {DRIVE_ZIP}')
elif not os.path.exists(DATA_DIR):
    from google.colab import files
    uploaded = files.upload()
    !unzip -q cosyvoice3_data.zip -d /content/
else:
    print(f'Data exists: {DATA_DIR}')

for split in ['train', 'dev']:
    d = os.path.join(DATA_DIR, split)
    if os.path.exists(d):
        wc = len([f for f in os.listdir(os.path.join(d,'wavs')) if f.endswith('.wav')])
        print(f'{split}: {wc} wavs')

---
## 3. CosyVoice Clone + 의존성 설치

Colab 시스템 Python/PyTorch를 그대로 사용합니다.
CosyVoice `requirements.txt`에서 이미 있거나 불필요한 것만 제외하고 설치합니다.

In [ ]:
%%bash
set -e

# === CosyVoice clone ===
cd /content
if [ ! -d "CosyVoice" ]; then
    git clone --depth 1 https://github.com/FunAudioLLM/CosyVoice.git
    cd CosyVoice && git submodule update --init --recursive
else
    echo "CosyVoice already cloned"
fi

# === path.sh ===
cd /content/CosyVoice
mkdir -p examples/libritts/cosyvoice3
cat > examples/libritts/cosyvoice3/path.sh << 'EOF'
export PYTHONPATH=/content/CosyVoice/third_party/Matcha-TTS:/content/CosyVoice:${PYTHONPATH:-}
EOF

echo "Done: $(ls -d /content/CosyVoice)"

In [ ]:
%%bash
set -e
cd /content/CosyVoice

echo "=== [1/7] 기본 도구 ==="
pip install setuptools wget

echo "=== [2/7] ML 핵심 ==="
pip install onnxruntime-gpu conformer hyperpyyaml "protobuf>=5.26,<6"

echo "=== [3/7] 학습 프레임워크 ==="
pip install "transformers>=4.45,<5" --force-reinstall
pip install lightning accelerate diffusers hydra-core deepspeed

echo "=== [4/7] 오디오 + 유틸 ==="
pip install soundfile librosa inflect pyarrow pydantic tiktoken more-itertools huggingface_hub pyworld

echo "=== [5/7] whisper + numba 호환 ==="
pip install --upgrade numba
pip install openai-whisper --no-deps

echo "=== [6/7] Matcha-TTS (Cython 빌드만) ==="
cd third_party/Matcha-TTS
pip install cython
python setup.py build_ext --inplace

echo ""
echo "=== [7/7] 버전 확인 ==="
python -c "import transformers; print(f'transformers {transformers.__version__}')"
python -c "import deepspeed; print(f'deepspeed {deepspeed.__version__}')"
echo "=== All Done ==="

In [ ]:
# 검증
import sys
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')
sys.path.insert(0, '/content/CosyVoice')

import torch; print(f'torch {torch.__version__} CUDA={torch.cuda.is_available()}')
import numpy as np; print(f'numpy {np.__version__}')
import pkg_resources; print('pkg_resources OK')
import onnxruntime as ort; print(f'onnxruntime {ort.__version__}')
import whisper; print('whisper OK')
import transformers; print(f'transformers {transformers.__version__}')
from hyperpyyaml import load_hyperpyyaml; print('HyperPyYAML OK')
import matcha; print('matcha OK')
from cosyvoice.dataset.processor import parquet_opener; print('cosyvoice.dataset OK')
print('\n=== All OK ===')

---
## 4. Pretrained 모델 다운로드

In [ ]:
import os
from huggingface_hub import snapshot_download

MODEL_DIR = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'

if not os.path.exists(MODEL_DIR) or not os.listdir(MODEL_DIR):
    os.makedirs(os.path.dirname(MODEL_DIR), exist_ok=True)
    snapshot_download(
        repo_id='FunAudioLLM/Fun-CosyVoice3-0.5B-2512',
        local_dir=MODEL_DIR,
        ignore_patterns=['*.md', '*.txt'],
    )
    print('Downloaded')
else:
    print(f'Exists: {MODEL_DIR}')

for f in ['llm.pt', 'flow.pt', 'hift.pt', 'campplus.onnx', 'speech_tokenizer_v3.onnx', 'cosyvoice3.yaml']:
    p = os.path.join(MODEL_DIR, f)
    s = f'{os.path.getsize(p)/1e6:.1f}MB' if os.path.exists(p) else 'MISSING'
    print(f'  {f}: {s}')
print(f'  CosyVoice-BlankEN/: {"OK" if os.path.isdir(os.path.join(MODEL_DIR, "CosyVoice-BlankEN")) else "MISSING"}')

---
## 5. wav.scp 경로 변환 + 데이터 링크

In [ ]:
import os

DATA_DIR = '/content/cosyvoice3_data'

for split in ['train', 'dev']:
    split_dir = os.path.join(DATA_DIR, split)
    scp_path = os.path.join(split_dir, 'wav.scp')

    with open(scp_path) as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split(' ', 1)
        if len(parts) != 2: continue
        utt_id, wav_path = parts
        if not wav_path.startswith('/'):
            wav_path = os.path.join(split_dir, wav_path)
        if '\\' in wav_path or (':' in wav_path and wav_path[1] == ':'):
            wav_path = os.path.join(split_dir, 'wavs', os.path.basename(wav_path))
        new_lines.append(f'{utt_id} {wav_path}\n')

    with open(scp_path, 'w') as f:
        f.writelines(new_lines)

    sample = new_lines[0].strip().split(' ', 1)[1]
    print(f'{split}: {len(new_lines)} entries, exists={os.path.exists(sample)}')

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3

mkdir -p data
ln -sfn /content/cosyvoice3_data/train data/train
ln -sfn /content/cosyvoice3_data/dev data/dev

wc -l data/train/wav.scp data/train/text data/dev/wav.scp data/dev/text
echo ''
head -1 data/train/text

In [ ]:
# instruct_text 생성 (text 파일을 복사)
# CosyVoice3 instruct 모드 학습 시 필요한 파일
# 없으면 parquet에 instruct_token이 빠지고, 학습 중 KeyError 발생
import shutil
DATA_DIR = '/content/cosyvoice3_data'
for split in ['train', 'dev']:
    src = os.path.join(DATA_DIR, split, 'text')
    dst = os.path.join(DATA_DIR, split, 'instruct_text')
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'{split}: instruct_text created')
    else:
        print(f'{split}: instruct_text exists')

---
## 6. Speaker Embedding 추출 (CAMPlus)

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"

for split in train dev; do
    echo "=== Embedding: $split ==="
    python ../../../tools/extract_embedding.py \
        --dir data/$split \
        --onnx_path $PRETRAINED/campplus.onnx \
        --num_thread 4
done

for split in train dev; do
    ls -lh data/$split/utt2embedding.pt data/$split/spk2embedding.pt 2>/dev/null || echo "$split: MISSING"
done

---
## 7. Speech Token 추출 (speech_tokenizer_v3)

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"

for split in train dev; do
    echo "=== Speech Token: $split ==="
    python ../../../tools/extract_speech_token.py \
        --dir data/$split \
        --onnx_path $PRETRAINED/speech_tokenizer_v3.onnx
done

for split in train dev; do
    ls -lh data/$split/utt2speech_token.pt 2>/dev/null || echo "$split: MISSING"
done

---
## 8. Parquet 변환

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

for split in train dev; do
    echo "=== Parquet: $split ==="
    mkdir -p data/$split/parquet
    python ../../../tools/make_parquet_list.py \
        --num_utts_per_parquet 1000 \
        --num_processes 4 \
        --src_dir data/$split \
        --des_dir data/$split/parquet
done

cat data/train/parquet/data.list > data/train.data.list
cat data/dev/parquet/data.list > data/dev.data.list

echo ''
echo 'train.data.list:'
cat data/train.data.list
echo 'dev.data.list:'
cat data/dev.data.list

---
## 9. Config 준비 + deepspeed 패치

In [ ]:
import os, shutil, re

TRAIN_BASE = '/content/CosyVoice/examples/libritts/cosyvoice3'
CONF_DIR = os.path.join(TRAIN_BASE, 'conf')
os.makedirs(CONF_DIR, exist_ok=True)

# cosyvoice3.yaml
cfg_dst = os.path.join(CONF_DIR, 'cosyvoice3.yaml')
cfg_src = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B/cosyvoice3.yaml'
if not os.path.exists(cfg_dst):
    shutil.copy2(cfg_src, cfg_dst)
    print(f'Config copied')

with open(cfg_dst) as f:
    for line in f:
        if 'max_epoch' in line or 'train_conf' in line or 'log_interval' in line:
            print(line.rstrip())

# deepspeed 패치: 모든 deepspeed 관련 import를 try/except로 감싸기
import glob
patched = []
for py_file in glob.glob('/content/CosyVoice/cosyvoice/**/*.py', recursive=True):
    with open(py_file) as f:
        lines = f.readlines()

    new_lines = []
    modified = False
    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()
        # import deepspeed 또는 from deepspeed... import ... 패턴
        if ('import deepspeed' in stripped or stripped.startswith('from deepspeed')) and 'try:' not in (lines[i-1].strip() if i > 0 else ''):
            indent = line[:len(line) - len(line.lstrip())]
            new_lines.append(f'{indent}try:\n')
            new_lines.append(f'{indent}    {stripped}\n')
            new_lines.append(f'{indent}except (ImportError, ModuleNotFoundError):\n')
            # from X import Y -> Y = None
            m = re.match(r'from\s+\S+\s+import\s+(.+)', stripped)
            if m:
                names = [n.strip() for n in m.group(1).split(',')]
                for name in names:
                    new_lines.append(f'{indent}    {name} = None\n')
            else:
                new_lines.append(f'{indent}    deepspeed = None\n')
            modified = True
        else:
            new_lines.append(line)
        i += 1

    if modified:
        with open(py_file, 'w') as f:
            f.writelines(new_lines)
        patched.append(py_file.replace('/content/CosyVoice/', ''))

if patched:
    print(f'deepspeed patched ({len(patched)} files):')
    for p in patched:
        print(f'  {p}')
else:
    print('deepspeed already patched')

---
## 9-2. Colab 호환 패치

CosyVoice 원본 코드가 Colab 환경에서 돌아가도록 필수 패치를 적용합니다.
- `instruct_text` 파일 생성 (instruct 모드 학습용)
- `llm.py`: instruct_token 누락 시 text_token fallback
- `train_utils.py`: cosyvoice_join 우회, prefetch_factor 조건부 처리
- `train.py`: bfloat16 dtype 통일, GradScaler 비활성화

In [ ]:
import re

# === 1. tokenizer.py: transformers 5.x 호환 ===
# AutoTokenizer는 5.x에서 slow tokenizer(merges.txt)를 무조건 로드하려 함
# PreTrainedTokenizerFast로 교체하면 tokenizer.json만 사용하므로 버전 무관하게 동작
tok_path = '/content/CosyVoice/cosyvoice/tokenizer/tokenizer.py'
with open(tok_path) as f:
    code = f.read()
# import 추가
if 'PreTrainedTokenizerFast' not in code:
    code = code.replace(
        'from transformers import AutoTokenizer',
        'from transformers import AutoTokenizer, PreTrainedTokenizerFast'
    )
# AutoTokenizer → PreTrainedTokenizerFast 교체
code = code.replace(
    'self.tokenizer = AutoTokenizer.from_pretrained(token_path)',
    'self.tokenizer = PreTrainedTokenizerFast.from_pretrained(token_path)'
)
# use_fast=True 버전도 교체 (이전 패치가 적용된 경우)
code = code.replace(
    'self.tokenizer = AutoTokenizer.from_pretrained(token_path, use_fast=True)',
    'self.tokenizer = PreTrainedTokenizerFast.from_pretrained(token_path)'
)
with open(tok_path, 'w') as f:
    f.write(code)

# === 2. llm.py: instruct_token 누락 시 text_token으로 fallback ===
llm_path = '/content/CosyVoice/cosyvoice/llm/llm.py'
with open(llm_path) as f:
    code = f.read()
code = code.replace("batch['instruct_token']", "batch.get('instruct_token', batch['text_token'])")
code = code.replace("batch['instruct_token_len']", "batch.get('instruct_token_len', batch['text_token_len'])")
with open(llm_path, 'w') as f:
    f.write(code)

# === 3. train_utils.py: cosyvoice_join 우회 + prefetch_factor 조건부 + bfloat16 ===
utils_path = '/content/CosyVoice/cosyvoice/utils/train_utils.py'
with open(utils_path) as f:
    code = f.read()
code = code.replace(
    'def cosyvoice_join(group_join, info_dict):',
    'def cosyvoice_join(group_join, info_dict):\n    return False'
)
code = code.replace(
    'prefetch_factor=args.prefetch',
    'prefetch_factor=args.prefetch if args.num_workers > 0 else None'
)
code = code.replace('dtype=dtype', 'dtype=torch.bfloat16')
code = code.replace('enabled=scaler is not None', 'enabled=True')
with open(utils_path, 'w') as f:
    f.write(code)

# === 4. train.py: bfloat16 모델 변환 + GradScaler 비활성화 ===
train_path = '/content/CosyVoice/cosyvoice/bin/train.py'
with open(train_path) as f:
    code = f.read()
code = code.replace('model.cuda()', 'model.cuda().bfloat16()')
code = code.replace(
    'scaler = torch.cuda.amp.GradScaler() if args.use_amp else None',
    'scaler = torch.cuda.amp.GradScaler(enabled=False) if args.use_amp else None'
)
with open(train_path, 'w') as f:
    f.write(code)

print('=== Colab patches applied ===')
print('  tokenizer.py: PreTrainedTokenizerFast (transformers 5.x 호환)')
print('  llm.py: instruct_token fallback')
print('  train_utils.py: cosyvoice_join bypass, prefetch fix, bfloat16')
print('  train.py: bfloat16 model, GradScaler disabled')

---
## 10. Dry Run (1 Epoch)

학습이 정상 동작하는지 확인합니다.

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh
export CUDA_VISIBLE_DEVICES="0"

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"

torchrun --nnodes=1 --nproc_per_node=1 \
    --rdzv_id=1986 --rdzv_backend="c10d" --rdzv_endpoint="localhost:1234" \
  ../../../cosyvoice/bin/train.py \
  --train_engine torch_ddp \
  --config conf/cosyvoice3.yaml \
  --train_data data/train.data.list \
  --cv_data data/dev.data.list \
  --qwen_pretrain_path $PRETRAINED/CosyVoice-BlankEN \
  --onnx_path $PRETRAINED \
  --model llm \
  --checkpoint $PRETRAINED/llm.pt \
  --model_dir `pwd`/exp/dryrun/llm/torch_ddp \
  --tensorboard_dir `pwd`/tensorboard/dryrun \
  --ddp.dist_backend nccl \
  --num_workers 2 \
  --prefetch 100 \
  --pin_memory \
  --use_amp \
  2>&1 | tail -80

echo ""
nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

---
## 11. 본 학습 (LLM SFT)

In [ ]:
import threading, time, shutil, os

stop_backup = threading.Event()

def auto_backup():
    EXP = '/content/CosyVoice/examples/libritts/cosyvoice3/exp/cosyvoice3/llm/torch_ddp'
    BAK = '/content/drive/MyDrive/cosyvoice3_backup/exp'
    while not stop_backup.is_set():
        time.sleep(300)
        if os.path.exists(EXP):
            try:
                pts = sorted([f for f in os.listdir(EXP) if f.endswith('.pt')],
                    key=lambda x: os.path.getmtime(os.path.join(EXP, x)), reverse=True)
                if pts:
                    os.makedirs(BAK, exist_ok=True)
                    dst = os.path.join(BAK, pts[0])
                    if not os.path.exists(dst):
                        shutil.copy2(os.path.join(EXP, pts[0]), dst)
                        print(f'[Backup] {pts[0]}')
            except Exception as e:
                print(f'[Backup Error] {e}')

threading.Thread(target=auto_backup, daemon=True).start()
print('Auto backup started (5min)')

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh
export CUDA_VISIBLE_DEVICES="0"

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"

torchrun --nnodes=1 --nproc_per_node=1 \
    --rdzv_id=1986 --rdzv_backend="c10d" --rdzv_endpoint="localhost:1234" \
  ../../../cosyvoice/bin/train.py \
  --train_engine torch_ddp \
  --config conf/cosyvoice3.yaml \
  --train_data data/train.data.list \
  --cv_data data/dev.data.list \
  --qwen_pretrain_path $PRETRAINED/CosyVoice-BlankEN \
  --onnx_path $PRETRAINED \
  --model llm \
  --checkpoint $PRETRAINED/llm.pt \
  --model_dir `pwd`/exp/cosyvoice3/llm/torch_ddp \
  --tensorboard_dir `pwd`/tensorboard/cosyvoice3/llm/torch_ddp \
  --ddp.dist_backend nccl \
  --num_workers 2 \
  --prefetch 100 \
  --pin_memory \
  --use_amp \
  2>&1 | tee /content/train_log.txt

In [ ]:
stop_backup.set()

import re, matplotlib.pyplot as plt, os

train_losses, cv_losses = [], []
if os.path.exists('/content/train_log.txt'):
    with open('/content/train_log.txt') as f:
        for line in f:
            m = re.search(r'train_loss[=:\s]+([\d.]+)', line)
            if m: train_losses.append(float(m.group(1)))
            m = re.search(r'cv_loss[=:\s]+([\d.]+)', line)
            if m: cv_losses.append(float(m.group(1)))

if train_losses:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(train_losses); axes[0].set_title('Train Loss'); axes[0].grid(True)
    if cv_losses:
        axes[1].plot(cv_losses, color='orange'); axes[1].set_title('CV Loss'); axes[1].grid(True)
    plt.tight_layout(); plt.show()
    print(f'Train: {train_losses[0]:.4f} -> {train_losses[-1]:.4f}')
    if cv_losses: print(f'CV: {cv_losses[0]:.4f} -> {cv_losses[-1]:.4f} (best: {min(cv_losses):.4f})')
else:
    print('No loss found')

---
## 12. Checkpoint 평균화 + 음성 평가

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

EXP_DIR="exp/cosyvoice3/llm/torch_ddp"
ls -lh $EXP_DIR/*.pt 2>/dev/null | tail -10

python ../../../cosyvoice/bin/average_model.py \
    --dst_model $EXP_DIR/llm.pt \
    --src_path $EXP_DIR \
    --num 5 \
    --val_best

In [ ]:
import sys, os, shutil
sys.path.insert(0, '/content/CosyVoice')
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')

PRETRAINED = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'
EXP_DIR = '/content/CosyVoice/examples/libritts/cosyvoice3/exp/cosyvoice3/llm/torch_ddp'

llm_orig = os.path.join(PRETRAINED, 'llm.pt')
llm_bak = os.path.join(PRETRAINED, 'llm.pt.bak')
if not os.path.exists(llm_bak):
    shutil.copy2(llm_orig, llm_bak)
shutil.copy2(os.path.join(EXP_DIR, 'llm.pt'), llm_orig)
print('Fine-tuned LLM applied')

from cosyvoice.cli.cosyvoice import CosyVoice3
cosyvoice = CosyVoice3(PRETRAINED, load_trt=False)
print('Model loaded')

In [ ]:
import os, soundfile as sf
from IPython.display import Audio, display

tests = [
    ("나한텐 일상이었어", "[excited]"),
    ("미안해, 다음엔 더 잘할게", "[sad]"),
    ("오~ 꽤하잖아!", "[happy]"),
    ("한숨 돌렸다 가자고. 시간은 충분하니까", "[calm]"),
]

os.makedirs('/content/test_outputs', exist_ok=True)

for i, (text, style) in enumerate(tests):
    print(f'\n--- {style} "{text}" ---')
    for spk in ['debi', 'marlene']:
        try:
            wavs = sorted([f for f in os.listdir('/content/cosyvoice3_data/train/wavs/') if f.startswith(spk)])
            if not wavs: continue
            ref = f'/content/cosyvoice3_data/train/wavs/{wavs[0]}'
            out = None
            for r in cosyvoice.inference_instruct2(
                f'{style}{text}', 'You are a helpful assistant.<|endofprompt|>', ref, stream=False
            ):
                out = r['tts_speech'].numpy()
            if out is not None:
                path = f'/content/test_outputs/test_{i+1}_{spk}_{style.strip("[]")}.wav'
                sf.write(path, out.flatten(), 24000)
                print(f'  {spk}: OK')
                display(Audio(out.flatten(), rate=24000))
        except Exception as e:
            print(f'  {spk}: {e}')

---
## 13. 저장 (Drive + HuggingFace)

In [ ]:
import shutil, os

PRETRAINED = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'
SAVE_DIR = '/content/drive/MyDrive/cosyvoice3_backup/final_model'
os.makedirs(SAVE_DIR, exist_ok=True)

for f in ['llm.pt', 'flow.pt', 'hift.pt', 'campplus.onnx', 'speech_tokenizer_v3.onnx', 'cosyvoice3.yaml']:
    src = os.path.join(PRETRAINED, f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(SAVE_DIR, f))
        print(f'  {f}: {os.path.getsize(src)/1e6:.1f}MB')

for d in os.listdir(PRETRAINED):
    src = os.path.join(PRETRAINED, d)
    if os.path.isdir(src):
        dst = os.path.join(SAVE_DIR, d)
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'  {d}/')

ref_dir = os.path.join(SAVE_DIR, 'references')
os.makedirs(ref_dir, exist_ok=True)
for spk in ['debi', 'marlene']:
    wavs = sorted([f for f in os.listdir('/content/cosyvoice3_data/train/wavs/') if f.startswith(spk)])
    if wavs:
        shutil.copy2(f'/content/cosyvoice3_data/train/wavs/{wavs[0]}', os.path.join(ref_dir, f'{spk}.wav'))

print(f'\nSaved: {SAVE_DIR}')

In [ ]:
from huggingface_hub import HfApi, login
login()

api = HfApi()
api.create_repo(repo_id='2R4mi/cosyvoice3-debi-marlene', exist_ok=True, private=True)
api.upload_folder(
    folder_path='/content/drive/MyDrive/cosyvoice3_backup/final_model',
    repo_id='2R4mi/cosyvoice3-debi-marlene',
    commit_message='CosyVoice3 fine-tuned: debi + marlene (LLM SFT)',
)
print('Uploaded')